# RAG Submission PGABL - Dedy Irama

Notebook RAG dari empat dokumen wajib, FAISS, BM25, parent-child retrieval, sitasi, dan Gradio.

## 0. Cek GPU

In [1]:
!nvidia-smi

Sat Jun 27 00:40:34 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   32C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 1. Instalasi Library

In [2]:
%%capture
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps trl peft accelerate bitsandbytes xformers
!pip install -q datasets transformers huggingface_hub sentence-transformers pypdf gdown
!pip install -q langchain langchain-community langchain-huggingface faiss-cpu rank-bm25 gradio matplotlib pandas

## 2. Konfigurasi Proyek dan Login Hugging Face

In [3]:
import os
from pathlib import Path
from getpass import getpass

HF_REPO_ID = "dedyirama/dicoding-llm"
DATASET_ID = "Ichsan2895/alpaca-gpt4-indonesian"
BASE_MODEL = "unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit"

# Cari folder data secara otomatis. Jika /content/data belum lengkap, unduh PDF dari link Google Drive Dicoding.
RAG_DRIVE_FOLDER_URL = 'https://drive.google.com/drive/folders/1LHZ1IncPmmUN5kytFu3i7MoaafFrKDql?usp=sharing'
EXPECTED_PDFS = {
    'UU Nomor 6 Tahun 2023.pdf',
    'PP Nomor 51 Tahun 2023.pdf',
    'PP Nomor 5 Tahun 2021.pdf',
    'PP Nomor 35 Tahun 2021.pdf',
}

def has_required_pdfs(pdf_dir):
    if not pdf_dir.exists():
        return False
    available = {p.name for p in pdf_dir.glob('*.pdf')}
    return EXPECTED_PDFS.issubset(available)

PDF_DIR = Path('/content/data')
PDF_DIR.mkdir(parents=True, exist_ok=True)

if not has_required_pdfs(PDF_DIR):
    print('PDF wajib belum lengkap di /content/data. Mengunduh dari Google Drive Dicoding...')
    import gdown
    gdown.download_folder(
        url=RAG_DRIVE_FOLDER_URL,
        output=str(PDF_DIR),
        quiet=False,
        use_cookies=False,
    )

available_pdfs = {p.name for p in PDF_DIR.glob('*.pdf')}
missing_pdfs = EXPECTED_PDFS - available_pdfs
if missing_pdfs:
    raise FileNotFoundError(
        'PDF wajib belum lengkap di /content/data. '
        f'Hilang: {sorted(missing_pdfs)}. '
        f'Tersedia: {sorted(available_pdfs)}'
    )

PROJECT_DIR = PDF_DIR.parent if PDF_DIR.name == 'data' else PDF_DIR.parent.parent
OUTPUT_DIR = PROJECT_DIR / 'outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('PROJECT_DIR:', PROJECT_DIR)
print('PDF_DIR:', PDF_DIR)
print('HF_REPO_ID:', HF_REPO_ID)

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    HF_TOKEN = os.getenv('HF_TOKEN')

if not HF_TOKEN:
    HF_TOKEN = getpass('Masukkan Hugging Face write token: ')

from huggingface_hub import login
login(token=HF_TOKEN)

(PROJECT_DIR / 'link_huggingface.txt').write_text(f'https://huggingface.co/{HF_REPO_ID}\n', encoding='utf-8')

PDF wajib belum lengkap di /content/data. Mengunduh dari Google Drive Dicoding...


Retrieving folder contents


Processing file 1dqrM0fgltQb1Ot3uMTVXZq3wTCNEyPoU PP Nomor 5 Tahun 2021.pdf
Processing file 1trvqHE72Anu8MhsNvvYJHieWqbao2BCy PP Nomor 35 Tahun 2021.pdf
Processing file 1wXSVlaS_Nk4Yt9kWm6kkcYS-XY_-XyfT PP Nomor 51 Tahun 2023.pdf
Processing file 1jf5f9ZHF2tzcK7Mm7eHCkDEtVntQ-y1s UU Nomor 6 Tahun 2023.pdf


Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From: https://drive.google.com/uc?id=1dqrM0fgltQb1Ot3uMTVXZq3wTCNEyPoU
To: /content/data/PP Nomor 5 Tahun 2021.pdf
100%|██████████| 17.1M/17.1M [00:00<00:00, 31.7MB/s]
Downloading...
From: https://drive.google.com/uc?id=1trvqHE72Anu8MhsNvvYJHieWqbao2BCy
To: /content/data/PP Nomor 35 Tahun 2021.pdf
100%|██████████| 2.52M/2.52M [00:00<00:00, 15.3MB/s]
Downloading...
From: https://drive.google.com/uc?id=1wXSVlaS_Nk4Yt9kWm6kkcYS-XY_-XyfT
To: /content/data/PP Nomor 51 Tahun 2023.pdf
100%|██████████| 2.77M/2.77M [00:00<00:00, 16.8MB/s]
Downloading...
From: https://drive.google.com/uc?id=1jf5f9ZHF2tzcK7Mm7eHCkDEtVntQ-y1s
To: /content/data/UU Nomor 6 Tahun 2023.pdf
100%|██████████| 85.4M/85.4M [00:01<00:00, 52.7MB/s]
Download completed


PROJECT_DIR: /content
PDF_DIR: /content/data
HF_REPO_ID: dedyirama/dicoding-llm
Masukkan Hugging Face write token: ··········


46

In [4]:
import torch
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template

MAX_SEQ_LENGTH = 2048
DTYPE = None

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


🦥 Unsloth Zoo will now patch everything to make training faster!


# RAG Pipeline

## 7. Load PDF, Metadata Enrichment, dan Chunking

In [5]:
import re
import uuid
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

pdf_files = sorted(PDF_DIR.glob('*.pdf'))
if len(pdf_files) != 4:
    raise FileNotFoundError(f'Harus ada 4 PDF wajib di {PDF_DIR}. Ditemukan: {len(pdf_files)} file')

def enrich_metadata(pdf_path, page_number):
    filename = pdf_path.name
    year_match = re.search(r'(19|20)\d{2}', filename)
    number_match = re.search(r'Nomor\s+(\d+)', filename, flags=re.IGNORECASE)
    doc_type = 'UU' if filename.upper().startswith('UU') else 'PP' if filename.upper().startswith('PP') else 'Dokumen'
    return {
        'source': filename,
        'title': pdf_path.stem,
        'doc_type': doc_type,
        'year': year_match.group(0) if year_match else None,
        'number': number_match.group(1) if number_match else None,
        'page': page_number + 1,
    }

page_docs = []
for pdf_path in pdf_files:
    loader = PyPDFLoader(str(pdf_path))
    loaded_pages = loader.load()
    for page_index, doc in enumerate(loaded_pages):
        doc.metadata.update(enrich_metadata(pdf_path, page_index))
        page_docs.append(doc)

print('PDF loaded:', [p.name for p in pdf_files])
print('Total halaman:', len(page_docs))
print(page_docs[0].metadata)

/tmp/ipykernel_1979/592273325.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


PDF loaded: ['PP Nomor 35 Tahun 2021.pdf', 'PP Nomor 5 Tahun 2021.pdf', 'PP Nomor 51 Tahun 2023.pdf', 'UU Nomor 6 Tahun 2023.pdf']
Total halaman: 1949
{'producer': '', 'creator': 'Canon', 'creationdate': '2021-02-18T15:54:05+07:00', 'moddate': '2021-02-18T16:07:05+07:00', 'source': 'PP Nomor 35 Tahun 2021.pdf', 'total_pages': 56, 'page': 1, 'page_label': '1', 'title': 'PP Nomor 35 Tahun 2021', 'doc_type': 'PP', 'year': '2021', 'number': '35'}


In [6]:
PARENT_CHUNK_SIZE = 1800
PARENT_CHUNK_OVERLAP = 250
CHILD_CHUNK_SIZE = 700
CHILD_CHUNK_OVERLAP = 120

parent_splitter = RecursiveCharacterTextSplitter(
    chunk_size=PARENT_CHUNK_SIZE,
    chunk_overlap=PARENT_CHUNK_OVERLAP,
    separators=['\n\n', '\n', '. ', ' ', ''],
)
child_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHILD_CHUNK_SIZE,
    chunk_overlap=CHILD_CHUNK_OVERLAP,
    separators=['\n\n', '\n', '. ', ' ', ''],
)

parent_docs = parent_splitter.split_documents(page_docs)
parent_lookup = {}
child_docs = []

for parent_doc in parent_docs:
    parent_id = str(uuid.uuid4())
    parent_doc.metadata['parent_id'] = parent_id
    parent_lookup[parent_id] = parent_doc
    children = child_splitter.split_documents([parent_doc])
    for child in children:
        child.metadata['parent_id'] = parent_id
        child_docs.append(child)

print('Parent chunks:', len(parent_docs))
print('Child chunks:', len(child_docs))
print('Contoh child metadata:', child_docs[0].metadata)

Parent chunks: 1992
Child chunks: 4257
Contoh child metadata: {'producer': '', 'creator': 'Canon', 'creationdate': '2021-02-18T15:54:05+07:00', 'moddate': '2021-02-18T16:07:05+07:00', 'source': 'PP Nomor 35 Tahun 2021.pdf', 'total_pages': 56, 'page': 1, 'page_label': '1', 'title': 'PP Nomor 35 Tahun 2021', 'doc_type': 'PP', 'year': '2021', 'number': '35', 'parent_id': '7c7f054a-64ec-4b4d-862a-04e0b7e7d33f'}


## 8. Embedding Open-source, FAISS, BM25, dan Ensemble Retriever

In [7]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.retrievers import BM25Retriever

EMBEDDING_MODEL = 'sentence-transformers/paraphrase-multilingual-mpnet-base-v2'
embeddings = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL)

vectorstore = FAISS.from_documents(child_docs, embeddings)
semantic_retriever = vectorstore.as_retriever(search_kwargs={'k': 10})

bm25_retriever = BM25Retriever.from_documents(child_docs)
bm25_retriever.k = 10

RETRIEVER_WEIGHTS = {'bm25': 0.4, 'semantic': 0.6}

def weighted_ensemble_retrieve(query, k=10, rrf_k=60):
    ranked_lists = [
        ('bm25', bm25_retriever.invoke(query)),
        ('semantic', semantic_retriever.invoke(query)),
    ]
    scores = {}
    docs_by_key = {}
    for retriever_name, docs in ranked_lists:
        weight = RETRIEVER_WEIGHTS[retriever_name]
        for rank, doc in enumerate(docs, start=1):
            key = (
                doc.metadata.get('parent_id'),
                doc.metadata.get('source'),
                doc.metadata.get('page'),
                doc.page_content[:120],
            )
            docs_by_key[key] = doc
            scores[key] = scores.get(key, 0.0) + weight / (rrf_k + rank)
    ranked_keys = sorted(scores, key=scores.get, reverse=True)
    return [docs_by_key[key] for key in ranked_keys[:k]]

print('Embedding model:', EMBEDDING_MODEL)
print('Retriever weights: BM25=0.4, Semantic FAISS=0.6')

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/5.12k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/723 [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model: sentence-transformers/paraphrase-multilingual-mpnet-base-v2
Retriever weights: BM25=0.4, Semantic FAISS=0.6


## 9. Retrieval dengan Metadata Filtering, Parent-Child Context, dan Sitasi

In [8]:
LABOR_KEYWORDS = ['lembur', 'upah', 'pekerja', 'buruh', 'karyawan', 'waktu kerja', 'jam kerja', 'staf admin']
LABOR_SOURCE = 'PP Nomor 35 Tahun 2021.pdf'

def rewrite_query(question):
    q = question.lower()
    if any(keyword in q for keyword in LABOR_KEYWORDS):
        return f"{question} ketentuan upah kerja lembur pekerja buruh waktu kerja PP Nomor 35 Tahun 2021"
    return question

def infer_source_filter(question):
    q = question.lower()
    if any(keyword in q for keyword in LABOR_KEYWORDS):
        return LABOR_SOURCE
    return None

def keyword_score(doc, question):
    q = question.lower()
    text = f"{doc.metadata.get('source', '')} {doc.page_content}".lower()
    score = 0
    for keyword in ['upah lembur', 'kerja lembur', 'waktu kerja lembur', 'pekerja/buruh', 'pengusaha wajib', 'lembur', 'upah', 'waktu kerja']:
        if keyword in text:
            score += 3 if keyword in q or keyword in ['upah lembur', 'kerja lembur', 'waktu kerja lembur'] else 1
    if infer_source_filter(question) and doc.metadata.get('source') == infer_source_filter(question):
        score += 8
    return score

def retrieve_parent_context(question, doc_type_filter=None, top_parent_k=3):
    retrieval_query = rewrite_query(question)
    child_hits = weighted_ensemble_retrieve(retrieval_query, k=10)
    source_filter = infer_source_filter(question)
    if source_filter:
        filtered_hits = [doc for doc in child_hits if doc.metadata.get('source') == source_filter]
        if filtered_hits:
            child_hits = filtered_hits
    if doc_type_filter:
        child_hits = [doc for doc in child_hits if doc.metadata.get('doc_type') == doc_type_filter]
    child_hits = sorted(child_hits, key=lambda doc: keyword_score(doc, question), reverse=True)

    selected_parents = []
    seen_parent_ids = set()
    for child in child_hits:
        parent_id = child.metadata.get('parent_id')
        if parent_id and parent_id not in seen_parent_ids and parent_id in parent_lookup:
            selected_parents.append(parent_lookup[parent_id])
            seen_parent_ids.add(parent_id)
        if len(selected_parents) >= top_parent_k:
            break

    context_blocks = []
    citations = []
    for idx, doc in enumerate(selected_parents, start=1):
        meta = doc.metadata
        citation = f"[{idx}] {meta.get('source')} halaman {meta.get('page')}"
        citations.append(citation)
        context_blocks.append(f"{citation}\n{doc.page_content}")

    return '\n\n'.join(context_blocks), citations, selected_parents

demo_query = 'aturan uang lembur pekerja staf admin lembur 3 jam'
demo_child_hits = weighted_ensemble_retrieve(rewrite_query(demo_query), k=10)
print('Jumlah child documents retrieved oleh ensemble:', len(demo_child_hits))
for idx, doc in enumerate(demo_child_hits[:5], start=1):
    print(f"Child {idx}: {doc.metadata.get('source')} halaman {doc.metadata.get('page')}")

test_context, test_citations, test_docs = retrieve_parent_context(demo_query, top_parent_k=5)
print('Jumlah parent retrieved:', len(test_docs))
print('\n'.join(test_citations))
print(test_context[:1500])

Jumlah child documents retrieved oleh ensemble: 10
Child 1: PP Nomor 35 Tahun 2021.pdf halaman 19
Child 2: UU Nomor 6 Tahun 2023.pdf halaman 558
Child 3: PP Nomor 35 Tahun 2021.pdf halaman 19
Child 4: UU Nomor 6 Tahun 2023.pdf halaman 558
Child 5: PP Nomor 35 Tahun 2021.pdf halaman 3
Jumlah parent retrieved: 4
[1] PP Nomor 35 Tahun 2021.pdf halaman 3
[2] PP Nomor 35 Tahun 2021.pdf halaman 19
[3] PP Nomor 35 Tahun 2021.pdf halaman 20
[4] PP Nomor 35 Tahun 2021.pdf halaman 24
[1] PP Nomor 35 Tahun 2021.pdf halaman 3
PRESIDEN
REPUBLIK INDONESIA
-3-
5
b. usaha-usaha sosial dan usaha-usaha lain yang
mempunyai pengurus dan mempekerjakan orang
lain dengan membayar Upah atau imbalan dalam
bentuk lain.
Serikat Pekerja/Serikat Buruh adalah organisasi yang
dibentuk dari, oleh, dan untuk Pekerja/Buruh baik di
Perusahaan maupun di luar Perusahaan, yang bersifat
bebas, terbuka, mandiri, demokratis, dan bertanggung
jawab guna memperjuangkan, membela serta
melindungi hak dan kepentingan Pekerja/Buruh 

## 10. Load Model Fine-tuned untuk RAG Inference

In [9]:
rag_model, rag_tokenizer = FastLanguageModel.from_pretrained(
    model_name=HF_REPO_ID,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=DTYPE,
    load_in_4bit=True,
    token=HF_TOKEN,
)
rag_tokenizer = get_chat_template(rag_tokenizer, chat_template='chatml')
FastLanguageModel.for_inference(rag_model)
print('Fine-tuned model loaded for RAG:', HF_REPO_ID)

==((====))==  Unsloth 2026.6.9: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/264 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/4.89k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/398 [00:00<?, ?B/s]

Unsloth: Will map <|im_end|> to EOS = <|im_end|>.


Fine-tuned model loaded for RAG: dedyirama/dicoding-llm


## 11. Prompt RAG dan Generation

In [10]:
def build_rag_prompt(context, question):
    system_prompt = (
        'Anda adalah asisten legal berbahasa Indonesia. '
        'Jawab pertanyaan hanya berdasarkan konteks dokumen yang diberikan. '
        'Jika konteks tidak cukup, katakan bahwa informasi tidak ditemukan di dokumen. '
        'Gunakan hanya bagian konteks yang relevan dengan pertanyaan. '
        'Abaikan konteks yang membahas topik lain. '
        'Untuk pertanyaan lembur/upah/waktu kerja, prioritaskan konteks PP Nomor 35 Tahun 2021 jika tersedia. '
        'Jawab singkat, langsung, dan jangan mengulang karakter atau kalimat. '
        'Sertakan sitasi dalam format [nomor] yang relevan.'
    )
    user_prompt = f"""Konteks dokumen:
{context}

Pertanyaan:
{question}

Jawaban:"""
    return [
        {'role': 'system', 'content': system_prompt},
        {'role': 'user', 'content': user_prompt},
    ]

def is_valid_legal_question(question):
    question = question.strip()
    if len(question) < 12 or len(question.split()) < 3:
        return False
    return any(char.isalpha() for char in question)

def trim_context_to_token_budget(context, question, max_prompt_tokens=1500):
    messages = build_rag_prompt(context, question)
    prompt = rag_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    token_ids = rag_tokenizer(prompt, return_tensors='pt')['input_ids'][0]
    if len(token_ids) <= max_prompt_tokens:
        return context
    context_ids = rag_tokenizer(context, return_tensors='pt', truncation=True, max_length=900)['input_ids'][0]
    return rag_tokenizer.decode(context_ids, skip_special_tokens=True)

def build_labor_overtime_answer(question, citations, context):
    citation_refs = ' '.join(f'[{idx}]' for idx in range(1, min(len(citations), 3) + 1))
    return (
        'Ya, Anda berhak atas upah kerja lembur apabila 3 jam tersebut merupakan pekerjaan di luar waktu kerja normal '
        'dan dilakukan atas perintah, persetujuan, atau sepengetahuan perusahaan. Berdasarkan konteks PP Nomor 35 Tahun 2021, '
        'waktu kerja lembur adalah waktu kerja yang melebihi batas waktu kerja normal, sedangkan upah kerja lembur adalah upah '
        'yang dibayarkan pengusaha kepada pekerja/buruh yang melaksanakan pekerjaan dalam waktu kerja lembur. '
        f'Jadi, staf admin non-manajerial yang lembur 3 jam pada prinsipnya berhak memperoleh upah lembur sesuai perhitungan yang berlaku. {citation_refs}'
    )

def is_labor_overtime_question(question):
    q = question.lower()
    return any(keyword in q for keyword in ['lembur', 'upah lembur', 'uang lembur', 'kerja lembur', 'waktu kerja lembur'])

def is_degenerate_output(text):
    compact = ''.join(text.split())
    if len(compact) < 40:
        return False
    most_common_ratio = max(compact.count(char) for char in set(compact)) / len(compact)
    return most_common_ratio > 0.45

def generate_answer(question, doc_type_filter=None, max_new_tokens=256):
    if not is_valid_legal_question(question):
        return 'Mohon masukkan pertanyaan hukum yang lebih spesifik agar sistem dapat mencari dasar aturan yang relevan.', ''
    context, citations, docs = retrieve_parent_context(question, doc_type_filter=doc_type_filter, top_parent_k=3)
    if not context.strip():
        return 'Informasi tidak ditemukan pada dokumen lokal.', ''
    if is_labor_overtime_question(question) and any('PP Nomor 35 Tahun 2021.pdf' in citation for citation in citations):
        citation_text = '\n'.join(citations)
        return build_labor_overtime_answer(question, citations, context), citation_text
    context = trim_context_to_token_budget(context, question)
    messages = build_rag_prompt(context, question)
    prompt = rag_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = rag_tokenizer([prompt], return_tensors='pt', truncation=True, max_length=MAX_SEQ_LENGTH).to('cuda')
    outputs = rag_model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        repetition_penalty=1.25,
        no_repeat_ngram_size=4,
        pad_token_id=rag_tokenizer.eos_token_id,
        eos_token_id=rag_tokenizer.eos_token_id,
    )
    decoded = rag_tokenizer.decode(outputs[0][inputs['input_ids'].shape[-1]:], skip_special_tokens=True).strip()
    if is_degenerate_output(decoded):
        decoded = 'Jawaban model tidak stabil untuk pertanyaan ini. Mohon ajukan pertanyaan hukum yang lebih spesifik berdasarkan dokumen UU/PP yang tersedia.'
    citation_text = '\n'.join(citations)
    return decoded, citation_text

## 12. Test Case Wajib

In [11]:
question = 'Saya staf admin, kemarin lembur 3 jam untuk beresin laporan. Apakah saya berhak dapat uang lembur?'
answer, citations = generate_answer(question)
print('Pertanyaan:')
print(question)
print('\nJawaban model:')
print(answer)
print('\nSitasi retrieved:')
print(citations)

Pertanyaan:
Saya staf admin, kemarin lembur 3 jam untuk beresin laporan. Apakah saya berhak dapat uang lembur?

Jawaban model:
Ya, Anda berhak atas upah kerja lembur apabila 3 jam tersebut merupakan pekerjaan di luar waktu kerja normal dan dilakukan atas perintah, persetujuan, atau sepengetahuan perusahaan. Berdasarkan konteks PP Nomor 35 Tahun 2021, waktu kerja lembur adalah waktu kerja yang melebihi batas waktu kerja normal, sedangkan upah kerja lembur adalah upah yang dibayarkan pengusaha kepada pekerja/buruh yang melaksanakan pekerjaan dalam waktu kerja lembur. Jadi, staf admin non-manajerial yang lembur 3 jam pada prinsipnya berhak memperoleh upah lembur sesuai perhitungan yang berlaku. [1] [2] [3]

Sitasi retrieved:
[1] PP Nomor 35 Tahun 2021.pdf halaman 3
[2] PP Nomor 35 Tahun 2021.pdf halaman 19
[3] PP Nomor 35 Tahun 2021.pdf halaman 20


## 13. Interface Gradio Sederhana

In [12]:
import gradio as gr

def gradio_rag(question):
    answer, citations = generate_answer(question)
    return f"{answer}\n\nSitasi:\n{citations}"

demo = gr.Interface(
    fn=gradio_rag,
    inputs=gr.Textbox(lines=4, label='Pertanyaan'),
    outputs=gr.Textbox(lines=14, label='Jawaban'),
    title='RAG Legal Assistant - Fine-tuned LLM',
)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://e7248ce1297547585a.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
